# **DATA EXPLORATION AND UNDERSTANDING**

## 1. Business Understanding

`INPUT VARIABLES`: numerical and categorical <br>
`OUPUT VARIABLE`: numeric (continuous variable) <br>

__GOAL__: Predict the price of cars

`carID` : An attribute that contains an identifier for each car <br>
`Brand` : The car's main brand number <br>
`Model` : The car model <br>
`year`: The year of Registration of the Car <br>
`mileage` : The total reported distance travelled by the car (in miles) <br>
`tax` : The amount of road tax that, in 2020, was applicable to the car in question <br>
`fuelType`: Type of Fuel used by the car <br>
`mpg`: Average Miles per Gallon <br>
`engineSize`: Size of Engine in liters (Cubic Decimeters) <br>
`paintQuality%`: The mechanic's assessment of the cars' overall paint quality and hull integrity (filled by the mechanic during evaluation) <br>
`previousOwners`: Number of previous registered owners of the vehicle <br>
`hasDamage`: Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not <br>
`price`: The car's price when purchased by Cars 4 You <br>

## 2. Imports and Read Data

In [ ]:
# importing the libraries
import pandas as pd
import numpy as np
from datetime import timedelta
from matplotlib import pyplot as plt
import math
import seaborn as sns
import re
import json
from collections import Counter

In [ ]:
# setting the options
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.max_rows', None)
#np.set_printoptions(threshold=np.inf)
#plt.style.use('seaborn-v0_8-dark')

In [ ]:
# importing the data
# df_training = pd.read_csv('C:/Users/calco/OneDrive/Ambiente de Trabalho/nova ims/1 Semestre/ML/project_data/train.csv', sep=',')
# df_test = pd.read_csv('C:/Users/calco/OneDrive/Ambiente de Trabalho/nova ims/1 Semestre/ML/project_data/test.csv', sep=',')

df_training = pd.read_csv('project_data/train.csv', sep=',')
df_test = pd.read_csv('project_data/test.csv', sep=',')


## 3. Macro-Inspection

In [ ]:
#Check the shape of the dataset
df_training.shape

In [ ]:
# check the first rows of the dataset
df_training.head()

In [ ]:
# check if there are any aggregation rows at the end of the dataset
df_training.tail()

In [ ]:
# check the datatypes and null counts
df_training.info()

Only **carID** and **price** don't have nan values

In [ ]:
#Check the data types of the features
df_training.dtypes

**Data type analysis:**

Features that should be booleans:
- **'hasDamage'**



In [ ]:
df_training.describe(include='all').T

## 4. Identifier Variable

This feature is the unique identifier of each claim - we will analyse this column in more depth in an attempt to assign it as the index of our dataframe.

In [ ]:
df_training[df_training['carID'].duplicated(keep=False)]

Upon review, there are no duplicated 'carID' values in the dataset. </br>
As we saw previously, this column has no null values, so we can set it as the dataframe index.

In [ ]:
df_training = df_training.set_index('carID').rename_axis(None)
df_training

In [ ]:
df_test = df_test.set_index('carID').rename_axis(None)

## 5. Missing Values

Has we saw previously only **carId** and the **price** don't have nana values

In [ ]:
# calculate the number of NaNs for each column
nan_counts = df_training.isna().sum()

# get the total number of rows
total_rows = df_training.shape[0]

# calculate the percentage of NaN values for each column
percentage_nans = round((nan_counts / total_rows) * 100,2)

# format the percentages with '%' sign
#percentage_nans = percentage_nans.apply(lambda x: f"{x:.2f}%")

# combine all the information into a DataFrame
nan_summary = pd.DataFrame({
    'NaN Count': nan_counts
    , 'Total Values': [total_rows] * len(nan_counts)
    , 'Percentage NaN(%)': percentage_nans
})

display(nan_summary)

By analyzing the output above, we can conclude:
- **'tax'** and **'mpg'**: approximately 10% of the data is missing for these features. 
- All other features have a minimal proportion of missing values, around 2%, which is not significant.

In [ ]:
#check if there is a row with all values are NaN
df_training[df_training.isna().all(axis=1)]

In [ ]:
nan_ratio = df_training.isna().mean(axis=1).sort_values()
nan_ratio

## 6. Unique Values

In [ ]:
df_training.nunique() #to see if there is more than 1 value in each feature

The feature **hasDamage** only has one type of value, meaning is a irrelavant feature

In [ ]:
# df_training.drop(columns='hasDamage', inplace = True)
# df_training

In [ ]:
#df_test.drop(columns='hasDamage', inplace=True)

## 7.Data Types
We will check each column for mixed data types and convert features to the appropriate types.

In [ ]:
# Transformar nomes das colunas
df_training.columns = [col.capitalize() for col in df_training.columns]
df_test.columns = [col.capitalize() for col in df_test.columns]
print(df_training.columns)

In [ ]:
df_training.rename(columns={
    'Fueltype': 'FuelType',
    'Enginesize': 'EngineSize',
    'Paintquality%': 'PaintQuality%',
    'Previousowners': 'PreviousOwners',
    'Mpg': 'MPG'
}, inplace=True)

df_test.rename(columns={
    'Fueltype': 'FuelType',
    'Enginesize': 'EngineSize',
    'Paintquality%': 'PaintQuality%',
    'Previousowners': 'PreviousOwners',
    'Mpg': 'MPG'
}, inplace=True)


In [ ]:
df_training.dtypes

In [ ]:
#check if there is any feature who has more than 1 dtype
def check_mixed_types(column):
    # if there are more than one unique data types (exclude NaN values, as these are considered floats)
    return len(set(column[pd.notna(column)].apply(type))) > 1

In [ ]:
mixed_type_columns = [col for col in df_training.columns if check_mixed_types(df_training[col])]
mixed_type_columns_test = [col for col in df_test.columns if check_mixed_types(df_test[col])]

print(mixed_type_columns)
print(mixed_type_columns_test)

In [ ]:
df_test.columns

## 8. Categorical Features

In [ ]:
# df_copy=df.copy()
# df_test_copy = df_test.copy()

In [ ]:
numerical_columns = df_training.select_dtypes(include=['number']).columns
categorical_columns = df_test.select_dtypes(include=['object']).columns
categorical_columns

In [ ]:
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df_training[col].unique())

In **brands**,**model**,, **transmission**, **fuel type**  we can see that are weird values and also names that are in capitall letter and low letter

In [ ]:
# lets first change every value to capital letter
df_training['Brand']=df_training['Brand'].str.upper()

#now lets change the assumning filling error, to the full name of the brand
brand_lists = {
    'MERCEDES': ['MERCEDE', 'ERCEDES', 'ERCEDE'],
    'VW': ['V', 'MW', 'W'],
    'AUDI': ['UDI', 'AUD', 'UD'],
    'FORD': ['FOR', 'ORD', 'OR'],
    'BMW': ['MW', 'BM'],
    'HYUNDAI': ['YUNDAI', 'YUNDA', 'HYUNDA'],
    'SKODA': ['KODA', 'SKOD', 'KOD'],
    'TOYOTA': ['TOYOT','OYOTA',],
    'OPEL': ['OPE','PEL','PE']
    
}

# Loop through the dictionary and replace
for standard, variations in brand_lists.items():
    df_training['Brand'] = df_training['Brand'].replace(variations, standard)
    df_test['Brand'] = df_test['Brand'].replace(variations, standard)
    
df_training['Brand'].unique()

In [ ]:
# lets change every value to capital letter
df_training['Model']=df_training['Model'].str.upper()
#there are also felling errors like '2 SERIES' and ' 2 Series'-> extra space problem
df_training['Model']=df_training['Model'].str.strip()

corrections = {
    'ARTEO':'ARTEON',#N
    'YET':'YETI',#I
    'AMARO':'AMAROK',#K
    'RAPI':'RAPID',#D
    'GOL':'GOLF',#F
    'T-RO':'T-ROC',#C
    'GALAX':'GALAXY',#Y
    'SUPER':'SUPERB',#B
    'TOUARE':'TOUAREG',#G
    'U':'UP',#P
    'I80':'I800',
    'T':'TT',
    'VIV':'VIVARO',
    'ADA':'ADAM'

}

df_training['Model'] = df_training['Model'].replace(corrections)
  
hyun_brand=df_training['Brand']=='HYUNDAI'
i_start = df_training['Model'].str.startswith('I', na=False)
len_2 = df_training['Model'].str.len() == 2
filter= hyun_brand & i_start & len_2

def clean_model(val):
    if not isinstance(val, str):
        return val
    elif  val == 'VERSO-SO' or val=='CALIFORNIAE':
        return val[:-1]
    if val.endswith(('CLAS','SERIE','-CROS','FOCU','YARI','AURI')):
        return val + 'S'
    if val.endswith(('-MA','HILU')):
        return val + 'X'
    if val.endswith(('CRUISE','VELOSTE','-H','TOURE','OUTDOO','ROOMSTE')) :
        return val + 'R'
    if val.endswith(('VERS','SCIROCC','CITIG','MONDE','AYG','POL')) :
        return val + 'O'
    if val.endswith(('EDG','CARAVELL','LIF','BEETL','ALLSPAC','SANTA F')) :
        return val + 'E'
    if val.endswith(('CUSTO',)):
        return val + 'M'
    if val.endswith(('CONNEC','ECOSPOR','PASSA',)):
        return val + 'T'   
    if val.endswith(('FIEST', 'CORS','ASTR','INSIGNI','ZAFIR','OCTAVI','SCAL','KON','MOKK','KUG','FABI','MERIV','COROLL')):
        return val + 'A' 
    if val.endswith(('TIGUA', 'TUCSO','TOURA','SHARA')):
        return val + 'N' 
    if val.endswith(('KARO', 'KAMI','IONI','KODIA')):
        return val + 'Q' 
    if val.endswith(('220', '230', '200')):
        return 'C'+ val 
    return val

df_training['Model'] = df_training['Model'].apply(clean_model)
df_training.loc[filter, 'Model'] = df_training.loc[filter, 'Model'] + '0'

df_training['Model'].unique()

In [ ]:
#for the test

df_test['Model']=df_test['Model'].str.upper()
#there are also felling errors like '2 SERIES' and ' 2 Series'-> extra space problem
df_test['Model']=df_test['Model'].str.strip()

df_test['Model'] = df_test['Model'].replace(corrections)
  
hyun_brand=df_test['Brand']=='HYUNDAI'
i_start = df_test['Model'].str.startswith('I', na=False)
len_2 = df_test['Model'].str.len() == 2
filter= hyun_brand & i_start & len_2

df_test['Model'] = df_test['Model'].apply(clean_model)
df_test.loc[filter, 'Model'] = df_test.loc[filter, 'Model'] + '0'


In [ ]:
df_training['FuelType']=df_training['FuelType'].str.upper()
df_training['FuelType']=df_training['FuelType'].str.strip()

FuelType_lists = { 
    'PETROL': ['ETROL', 'ETRO','PETRO'],
    'DIESEL': ['DIESE', 'IESE', 'IESEL'],
    'HYBRID': ['YBRID', 'HYBRI','YBRI'],
    'OTHER': ['THER','OTHE']
}

for new, old in FuelType_lists.items():
    df_training['FuelType']=df_training['FuelType'].replace(old, new)
    df_test['FuelType']=df_test['FuelType'].replace(old, new)

df_training['FuelType'].unique()

In [ ]:
#Other doesn't make much sense so I will turn all the Other to unknown values

df_training.loc[df_training['FuelType']=='OTHER','FuelType']='UNKNOWN'

df_training['FuelType'].unique()

In [ ]:
df_training['Transmission']=df_training['Transmission'].str.upper()
df_training['Transmission']=df_training['Transmission'].str.strip()

transmission_list = {
    'SEMI-AUTOMATIC': ['SEMI-AUT','SEMI-AUT','EMI-AUTO','SEMI-AUTO','EMI-AUT'],
    'MANUAL': ['ANUAL','MANUA','ANUA'],
    'AUTOMATIC': ['AUTOMATI', 'UTOMATIC','UTOMATI'],
    'UNKNOWN': ['NKNOWN', 'NKNOW', 'UNKNOW']
}

for word, value in transmission_list.items():
    df_training['Transmission'] = df_training['Transmission'].replace(value, word)
    df_test['Transmission'] = df_test['Transmission'].replace(value, word)

df_training['Transmission'].unique()

In [ ]:
#Same with transmission_list other doesn't make much sense so I will turn all the Other to unknown values

df_training.loc[df_training['Transmission']=='OTHER','Transmission']='UNKNOWN'

df_training['Transmission'].unique()

In [ ]:
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df_training[col].unique())

## 9.Numerical Features

In [ ]:
numerical_columns

In [ ]:
for col in numerical_columns:
    print(f"\n--- {col} ---")
    print(df_training[col].unique())

We can see that there are fetures that have negative numbers which doesn't make sense

In [ ]:
for col in numerical_columns:
    df_training.loc[df_training[col] < 0, col] = np.nan
for col in numerical_columns:
    if col!='Price':
        df_test.loc[df_test[col] < 0, col] = np.nan

Year and Previous Owners has float numbers wich should just have integer, so I will round all the numbers to have 0 decimal values

In [ ]:
df_training['Year']=df_training['Year'].round()
df_training['PreviousOwners']=df_training['PreviousOwners'].round()
df_test['Year']=df_test['Year'].round()
df_test['PreviousOwners']=df_test['PreviousOwners'].round()

To be easier to read and also to reduse noise we will round the rest to 2 decimals

In [ ]:
col_remove=['Year', 'PreviousOwners']
new_numerical_columns=[i  for i in numerical_columns if i not in col_remove]
for col in new_numerical_columns:
    df_training[col]= df_training[col].round(2)
for col in new_numerical_columns:
    if col!="Price" :
        df_test[col]= df_test[col].round(2)

**PaintQuality%** values greater than 100% do not make sense in this context, because we assume that 100% represents perfect overall paint quality and hull integrity of the car

In [ ]:
df_training.loc[df_training['PaintQuality%'] > 100] = np.nan
df_test.loc[df_test['PaintQuality%'] > 100]  = np.nan

In [ ]:
inconsistent_col = numerical_columns

for col in inconsistent_col:
    print(f'---{col}---')
    print(df_training[col][~df_training[col].apply(lambda x: isinstance(x, (int, float)))])


In [ ]:
df_training.describe(include='all').T

## 10. Duplicated Rows

In [ ]:
#check if there is duplicated rows
duplicates = df_training[df_training.duplicated(keep=False)]
sorted_duplicates = duplicates.sort_values(by=list(df_training.columns))

sorted_duplicates

In [ ]:
df_training = df_training.drop_duplicates(keep='first')
df_test = df_test.drop_duplicates(keep='first')
df_test.shape

## Fill Missing Values

### Rows with all values equals to NaN

In [ ]:
df_training.isna().sum()

In [ ]:
rows_with_all_nan = df_training.isna().all(axis=1)

num_rows = rows_with_all_nan.sum()
print(f"Exists {num_rows} row/s completely none.")

if num_rows > 0:
    print(df_training[rows_with_all_nan])


In [ ]:
df_training = df_training.dropna(how='all')

### Brand

Firstly, let's see the NaN **Brands Models**, so we can fill them based on their model.

In [ ]:
brand_missing = df_training[df_training['Brand'].isna() & df_training['Model'].notna()]
brand_missing

Let's check if we have for each **Model**, multiples **Brand**.

In [ ]:
model_brand_counts = (
    df_training.groupby('Model')['Brand']
    .nunique()  
    .reset_index()
    .rename(columns={'Brand': 'Unique_Brands'})
)
models_with_multiple_brands = model_brand_counts[model_brand_counts['Unique_Brands'] > 1]
print(models_with_multiple_brands)

Lets check each brands to see if there are mistakes

In [ ]:
for model in models_with_multiple_brands['Model']:
    brand=df_training[df_training['Model']==model]['Brand'].unique()
    print(f"Model: {model} has this brands: {brand}")

As we can see, all the models that have more than one brand, the brands are BMW and Volkswagen. However, all of these models actually belong only to BMW.

In [ ]:
#let's change the brand of the previous models from vw to bmw 
for model in models_with_multiple_brands['Model']:
    filter= (df_training['Model']==model) & (df_training['Brand']=='VW')
    df_training.loc[filter,'Brand']='BMW'

   

In [ ]:
#checking if there is no more brands that have nore then one brand
model_brand_counts = (
    df_training.groupby('Model')['Brand']
    .nunique()  
    .reset_index()
    .rename(columns={'Brand': 'Unique_Brands'})
)
models_with_multiple_brands = model_brand_counts[model_brand_counts['Unique_Brands'] > 1]
print(models_with_multiple_brands)

In [ ]:
#make a dictionary of all brands based on the models (keys)
model_to_brand = df_training.groupby('Model')['Brand'].first().to_dict()
print(f"before(n? of missing values in brands): {df_training['Brand'].isna().sum()}")
# for each Brand it will fill the nan brands based on the model the car has(if the model is not nan)
df_training['Brand'] = df_training.apply(
    lambda row: model_to_brand[row['Model']] if (pd.isna(row['Brand']) and pd.notna(row['Model']))else row['Brand'],
    axis=1
)
print(f"after((n? of missing values in brands):  {df_training['Brand'].isna().sum()}")


Now, the only remaining missing **Brand** values are those where the **Model** is also missing.

### Model

Since the model is more specific within a brand, we will determine it basedbased on the mode of the car's year, brand, fuel type, engine size, and average miles per gallon. 

In [ ]:
df_training = df_training.copy()

print(f"before(n? of missing values in modles):  {df_training['Model'].isna().sum()}")

brand_to_model = df_training.groupby( ['Brand','Year', 'FuelType', 'EngineSize', 'MPG'])['Model'] \
                            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None) \
                            .to_dict()

df_training['Model'] = df_training.apply(
    lambda row: brand_to_model.get((row['Brand'], row['Year'],row['FuelType'],row['EngineSize'],row['MPG']), row['Model']) 
    if pd.isna(row['Model']) 
       and pd.notna(row['Brand']) 
       and pd.notna(row['Year']) 
       and pd.notna(row['FuelType']) 
       and pd.notna(row['EngineSize']) 
       and pd.notna(row['MPG'])
    else row['Model'],
    axis=1
)


print(f"after(n? of missing values in modles):  {df_training['Model'].isna().sum()}")


We still have 349 rows where **Model** are missing values.



### FuelType

Since the same brand and model in the same year usally have the same Fueltype, we will fill the NaN values with the mode based on the brand, model and yaer

In [ ]:
df_training = df_training.copy()

print(f"before(n? of missing values in modles):  {df_training['FuelType'].isna().sum()}")

brand_to_model = df_training.groupby( ['Brand','Year', 'Model'])['FuelType'] \
                            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None) \
                            .to_dict()

df_training['FuelType'] = df_training.apply(
    lambda row: brand_to_model.get((row['Brand'], row['Year'],row['Model']), row['FuelType']) 
    if pd.isna(row['FuelType']) 
       and pd.notna(row['Brand']) 
       and pd.notna(row['Year']) 
       and pd.notna(row['Model']) 
    else row['Model'],
    axis=1
)


print(f"Missing FuelType after filling: {df_training['FuelType'].isna().sum()}")


### For alll the categorical missing values

For the remaining categorical values, filling in Transmission is very ambiguous, as the same brand and model in the same year can have different types of Transmission, for example FORD Mustang (GT) in 2019 can be automatic and manual.

Similarly, filling the remaning NaN values of Brand, Model and FuelType would also be ambiguous, so we will replace these missing values with Unknown.

In [ ]:
categorical_cols = df_training.select_dtypes(include='object').columns

for col in categorical_cols:
    if df_training[col].isna:               
        df_training.loc[df_training[col].isna(),col]='UNKNOWN'
df_training.isna().sum()

### Numerical Features

For numerical features, we will use the median to fill in the missing values, based on the brand and model. We chose the median because unknown brands and models can have a wide range of dispersed values, and the median is less affected by outliers.

In [ ]:
df_training = df_training.copy()


numeric_cols = ['Year', 'Mileage', 'Tax', 'MPG', 'EngineSize', 'PaintQuality%', 'PreviousOwners', 'Hasdamage']


for col in numeric_cols:
    group_median = df_training.groupby(['Brand', 'Model'])[col].transform('mean')
    df_training[col] = df_training[col].fillna(group_median)
    #cause the Hyundai accent is the only car like that and has Mileage NaN
    brand_median = df_training.groupby('Brand')[col].transform('median')
    df_training[col] = df_training[col].fillna(brand_median)
    
 




### Categorical Features

For categorical features we will use the mode to fill missing values.

In [ ]:
df_training.isna().sum()

## 11. Export Dataset

In [ ]:
df_copy.to_excel("DataExplorationTrain.xlsx", index=False)


In [ ]:
df_test_copy.to_excel("DataExplorationTest.xlsx", index=False)
